In [56]:
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
import re
import string

In [57]:
df_true=pd.read_csv("C:\\Users\\HP\\Downloads\True.csv")
df_fake=pd.read_csv("C:\\Users\\HP\\Downloads\\Fake.csv")

In [58]:
df_true.head()

,title,text,subject,date
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,politicsNews,"December 31, 2017"
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,politicsNews,"December 29, 2017"
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,politicsNews,"December 31, 2017"
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,politicsNews,"December 30, 2017"
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,politicsNews,"December 29, 2017"


In [59]:
df_fake["class"]=0
df_true["class"]=1

In [60]:
df_merge=pd.concat([df_fake,df_true],axis=0)
df_merge.head(10)

,title,text,subject,date,class
0,Donald Trump Sends Out Embarrassing New Year’...,Donald Trump just couldn t wish all Americans ...,News,"December 31, 2017",0
1,Drunk Bragging Trump Staffer Started Russian ...,House Intelligence Committee Chairman Devin Nu...,News,"December 31, 2017",0
2,Sheriff David Clarke Becomes An Internet Joke...,"On Friday, it was revealed that former Milwauk...",News,"December 30, 2017",0
3,Trump Is So Obsessed He Even Has Obama’s Name...,"On Christmas day, Donald Trump announced that ...",News,"December 29, 2017",0
4,Pope Francis Just Called Out Donald Trump Dur...,Pope Francis used his annual Christmas Day mes...,News,"December 25, 2017",0
5,Racist Alabama Cops Brutalize Black Boy While...,The number of cases of cops brutalizing and ki...,News,"December 25, 2017",0
6,"Fresh Off The Golf Course, Trump Lashes Out A...",Donald Trump spent a good portion of his day a...,News,"December 23, 2017",0
7,Trump Said Some INSANELY Racist Stuff Inside ...,In the wake of yet another court decision that...,News,"December 23, 2017",0
8,Former CIA Director Slams Trump Over UN Bully...,Many people have raised the alarm regarding th...,News,"December 22, 2017",0
9,WATCH: Brand-New Pro-Trump Ad Features So Muc...,Just when you might have thought we d get a br...,News,"December 21, 2017",0


In [61]:
df=df_merge.drop(["title","subject","date"],axis=1)

In [62]:
df.shape

(44898, 2)

In [63]:
df.isnull().sum()

text     0
class    0
dtype: int64

In [64]:
df.duplicated().sum()

np.int64(6251)

In [65]:
df.drop_duplicates(inplace=True)

In [66]:
df.duplicated().sum()

np.int64(0)

In [67]:
df=df.sample(frac=1)
df

,text,class
11722,CAIRO (Reuters) - Islamic State has claimed an...,1
3035,Donald Trump may deny that he hired some Russi...,0
312,Donald Trump is getting absolutely hammered fo...,0
3379,Members of the Electoral College will not be r...,0
15724,HONOLULU (Reuters) - President Donald Trump ar...,1
...,...,...
11279,WASHINGTON/MOSCOW (Reuters) - The United State...,1
380,WASHINGTON (Reuters) - When Republicans tried ...,1
12274,NAIROBI (Reuters) - In the run-up to Kenya s A...,1
3977,WASHINGTON/NEW YORK (Reuters) - Congress has a...,1


In [68]:
df.reset_index(inplace=True)
df.drop(["index"],axis=1,inplace=True)

In [69]:
df.columns

Index(['text', 'class'], dtype='object')

In [70]:
df.head()

,text,class
0,CAIRO (Reuters) - Islamic State has claimed an...,1
1,Donald Trump may deny that he hired some Russi...,0
2,Donald Trump is getting absolutely hammered fo...,0
3,Members of the Electoral College will not be r...,0
4,HONOLULU (Reuters) - President Donald Trump ar...,1


In [71]:
def wordopt(text):
    text = text.lower()
    text = re.sub('\[.*?\]', '', text) #remove content within square braces
    text = re.sub("\\W"," ",text) # replace the non word charectors
    text = re.sub('https?://\S+|www\.\S+', '', text) # remove urls 
    text = re.sub('<.*?>+',   '', text) # remove HTML tags
    text = re.sub('[%s]' % re.escape(string.punctuation), '', text) # Remove the punctuations 
    text = re.sub('\n', '', text) # remove newline charectors
    text = re.sub('\w*\d\w*', '', text)   # remove words containing no  
    return text

In [72]:
df["text"]=df["text"].apply(wordopt)

In [73]:
x=df["text"]
y=df["class"]

In [75]:
x_train,x_test,y_train,y_test= train_test_split(x,y,test_size=0.25)

In [76]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorization = TfidfVectorizer()
xv_train = vectorization.fit_transform(x_train)
xv_test = vectorization.transform(x_test)


In [77]:
from sklearn.linear_model import LogisticRegression
LR= LogisticRegression()
LR.fit(xv_train,y_train)

LogisticRegression()

In [78]:
pred_lr=LR.predict(xv_test)

In [79]:
LR.score(xv_test,y_test)

0.9831297867936245

In [84]:
def predict_news(news_text):
    cleaned_text=wordopt(news_text)
    vectorized_text=vectorization.transform([cleaned_text])
    prediction= LR.predict(vectorized_text)
    return prediction[0]

In [85]:
news= "HELLLoooo i am"
prediction= predict_news(news)

if prediction == 1 :
    print("True")
else:
    print("False")

False
